In [18]:
# Install rdecision
install.packages("rdecision")

# Install CEdecisiontree
install.packages("CEdecisiontree")

# Install heemod
install.packages("heemod")

# Install igraph (often used with rdecision for plotting)
install.packages("igraph")

# Install DiagrammeR (often used with rdecision for plotting)
install.packages("DiagrammeR")

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

Warning message:
“package ‘CEdecisiontree’ is not available for this version of R

A version of this package for your version of R might be available elsewhere,
see the ideas at
https://cran.r-project.org/doc/manuals/r-patched/R-admin.html#Installing-packages”
also installing the dependencies ‘RcppArmadillo’, ‘mvnfast’


Warning message in install.packages("heemod"):
“installation of package ‘RcppArmadillo’ had non-zero exit status”
Warning message in install.packages("heemod"):
“installation of package ‘mvnfast’ had non-zero exit status”
Warning message in install.packages("heemod"):
“installation of package ‘heemod’ had non-zero exit status”
Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

Warning message in install.packages("igraph"):
“installation of package ‘igraph’ had non-zero exit status”
Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

In [1]:
library(rdecision)

# Define leaf nodes (terminal outcomes)
leaf_A <- LeafNode$new("Outcome A", utility = 0.8, cost = 100)
leaf_B <- LeafNode$new("Outcome B", utility = 0.9, cost = 50)
leaf_C <- LeafNode$new("Outcome C", utility = 0.7, cost = 120)
leaf_D <- LeafNode$new("Outcome D", utility = 1.0, cost = 20)
leaf_E <- LeafNode$new("Outcome E", utility = 0.6, cost = 150)
leaf_F <- LeafNode$new("Outcome F", utility = 0.95, cost = 30)

# Define chance nodes and their reactions (branches)
chance_test_pos_disease_status <- ChanceNode$new("Disease Status after Pos Test")
reaction_pos_disease <- Reaction$new(chance_test_pos_disease_status, leaf_A, p = 0.9, label = "Has Disease")
reaction_pos_no_disease <- Reaction$new(chance_test_pos_disease_status, leaf_B, p = 0.1, label = "No Disease")

chance_test_neg_disease_status <- ChanceNode$new("Disease Status after Neg Test")
reaction_neg_disease <- Reaction$new(chance_test_neg_disease_status, leaf_C, p = 0.2, label = "Has Disease")
reaction_neg_no_disease <- Reaction$new(chance_test_neg_disease_status, leaf_D, p = 0.8, label = "No Disease")

chance_test_result <- ChanceNode$new("Test Result")
reaction_test_pos <- Reaction$new(chance_test_result, chance_test_pos_disease_status, p = 0.3, label = "Positive Test", cost = 75) # Cost of test
reaction_test_neg <- Reaction$new(chance_test_result, chance_test_neg_disease_status, p = 0.7, label = "Negative Test")

chance_std_care_disease_status <- ChanceNode$new("Disease Status Standard Care")
reaction_std_disease <- Reaction$new(chance_std_care_disease_status, leaf_E, p = 0.5, label = "Has Disease")
reaction_std_no_disease <- Reaction$new(chance_std_care_disease_status, leaf_F, p = 0.5, label = "No Disease")

# Define decision node and its actions (strategies)
decision_node <- DecisionNode$new("Intervention Choice")
action_new_test <- Action$new(decision_node, chance_test_result, label = "New Test")
action_std_care <- Action$new(decision_node, chance_std_care_disease_status, label = "Standard Care")

# Create the decision tree
my_decision_tree <- DecisionTree$new(decision_node)

# Evaluate the tree
eval_results <- my_decision_tree$evaluate()
print(eval_results)

# To visualize (requires 'igraph' and 'DiagrammeR' to be installed)
# plot(my_decision_tree)

ERROR: Error in initialize(...): unused argument (cost = 100)


In [2]:
library(CEdecisiontree)

# Define nodes:
# 1: Decision (Treat/No Treat)
# 2: Chance - Success (from Treat)
# 3: Chance - Deterioration (from No Treat)
# 4: Leaf - Treated Success
# 5: Leaf - Treated Failure
# 6: Leaf - No Treat Deteriorates
# 7: Leaf - No Treat No Deterioration

# Transition probabilities (from rows to columns)
probs_matrix <- matrix(NA, nrow = 7, ncol = 7)
probs_matrix[1, 2] <- 0.6  # From Decision (1) to Treat branch (2) - This would typically be 1 if it's a decision, but CEdecisiontree structure is flexible
probs_matrix[1, 3] <- 0.4  # From Decision (1) to No Treat branch (3) - These are effectively strategy probabilities if treated as such

probs_matrix[2, 4] <- 0.8  # From Treat Success chance (2) to Treated Success Leaf (4)
probs_matrix[2, 5] <- 0.2  # From Treat Success chance (2) to Treated Failure Leaf (5)

probs_matrix[3, 6] <- 0.3  # From No Treat Deterioration chance (3) to No Treat Deteriorates Leaf (6)
probs_matrix[3, 7] <- 0.7  # From No Treat Deterioration chance (3) to No Treat No Deterioration Leaf (7)

# Costs (associated with reaching the nodes/branches, or terminal nodes)
cost_matrix <- matrix(0, nrow = 7, ncol = 7) # Initialize with 0

# Costs associated with transitions/states. Adjust based on how you define costs.
# For simplicity, let's assign costs to the terminal pathways.
# Note: CEdecisiontree often works by 'folding back' costs/utilities from leaves.
# Here, we'll assign "costs" to the transition *to* the terminal states.
cost_matrix[2, 4] <- 500  # Cost of reaching Treated Success
cost_matrix[2, 5] <- 1500 # Cost of reaching Treated Failure
cost_matrix[3, 6] <- 1000 # Cost of reaching No Treat Deteriorates
cost_matrix[3, 7] <- 200  # Cost of reaching No Treat No Deterioration

# Utilities (QALYs) - assign to terminal pathways or as per cycle values
utility_matrix <- matrix(0, nrow = 7, ncol = 7)

utility_matrix[2, 4] <- 0.9  # Utility of reaching Treated Success
utility_matrix[2, 5] <- 0.5  # Utility of reaching Treated Failure
utility_matrix[3, 6] <- 0.3  # Utility of reaching No Treat Deteriorates
utility_matrix[3, 7] <- 0.8  # Utility of reaching No Treat No Deterioration

# Define the model using the matrices
# The 'define_model' function expects lists for 'prob' and 'vals'
my_model_ce <- define_model(
  transmat = list(prob = probs_matrix, vals = cost_matrix),
  utility = list(prob = probs_matrix, vals = utility_matrix) # Pass utility matrix separately
)

# Calculate expected values (folding back)
expected_costs <- dectree_expected_values(model = my_model_ce$transmat)
expected_utilities <- dectree_expected_values(model = my_model_ce$utility)

print("Expected Costs per node:")
print(expected_costs)
print("Expected Utilities per node:")
print(expected_utilities)

# The 'run_cedectree' function can integrate this for CE analysis
# Need to set up 'create_ce_tree_long_df' for run_cedectree
# This is a more complex setup for full CE analysis, often involving
# defining "strategies" explicitly in a long format.

# Simplified view for demonstration:
# If node 1 is the decision node, its expected value reflects the overall strategy.
# You would compare expected_costs[1] and expected_utilities[1] (or whichever
# node represents the start of the comparative strategies).

ERROR: Error in library(CEdecisiontree): there is no package called ‘CEdecisiontree’


In [3]:
library(heemod)

# Define states (which effectively act as terminal nodes for a decision tree)
# We give them a "cost" and "utility" for a single "cycle" (representing the outcome)

state_treated_responder <- define_state(
  cost = 500,
  utility = 0.9
)

state_treated_non_responder <- define_state(
  cost = 1500,
  utility = 0.5
)

state_placebo_progresses <- define_state(
  cost = 1000,
  utility = 0.3
)

state_placebo_no_progression <- define_state(
  cost = 200,
  utility = 0.8
)

# Define transitions for each "strategy" (mimicking branches from decision node)

# Strategy 1: Active Treatment
# This transition matrix represents the probabilities from an implicit starting point
# to the two outcomes (responders/non-responders).
# Since it's a decision tree, we can consider this as a single "cycle"
# from an initial state to the terminal outcomes.
transition_active_treatment <- define_transition(
  0, 0, # Not used for this simple direct branch
  .5, .5 # P_responder, P_non_responder (simplified, usually from a prior chance node)
)

# Strategy 2: Placebo
transition_placebo <- define_transition(
  0, 0, # Not used
  .3, .7 # P_progresses, P_no_progression
)

# Create strategies
# For heemod, a "strategy" groups states and transitions.
# For a decision tree, each "strategy" effectively defines one path from the decision node.
# We define an "initial state" for each strategy and define probabilities of
# transitioning to the terminal "states".

# Let's adjust the heemod approach slightly to fit decision tree concept.
# We will define a "starting state" for each strategy, and from there,
# define transitions to the outcomes.

# Define a "dummy" initial state for active treatment
state_initial_active <- define_state(
  cost = 0, # Cost of starting
  utility = 0 # Utility of starting
)

# Define a "dummy" initial state for placebo
state_initial_placebo <- define_state(
  cost = 0,
  utility = 0
)

# Now, define the transition *from* these initial states *to* the outcome states
# For heemod, you list states and their transitions.
# The `define_transition` takes a matrix or sequence of probabilities.
# For a simple decision tree, we can define direct transitions.

# Strategy for Active Treatment
active_treatment_strategy <- define_strategy(
  transition = define_transition(
    # From 'state_initial_active' to the outcome states
    c(1, 0, 0, 0, # Prob to stay in initial (1), or go to other outcomes (0)
      0.6, 0.4, 0, 0, # From initial_active: 60% to responder, 40% to non-responder. Assuming a single step.
      0, 0, 1, 0,
      0, 0, 0, 1)
  ),
  state_initial_active,
  state_treated_responder,
  state_treated_non_responder,
  state_placebo_progresses, # Include all possible terminal states
  state_placebo_no_progression
)


# Strategy for Placebo
placebo_strategy <- define_strategy(
  transition = define_transition(
    # From 'state_initial_placebo' to the outcome states
    c(1, 0, 0, 0,
      0, 0, 0.3, 0.7, # From initial_placebo: 30% to progresses, 70% to no progression
      0, 0, 1, 0,
      0, 0, 0, 1)
  ),
  state_initial_placebo,
  state_treated_responder, # Include all possible terminal states
  state_treated_non_responder,
  state_placebo_progresses,
  state_placebo_no_progression
)


# Run the model
# For a decision tree, you typically run for cycles = 1 (one step from decision to outcome)
res_heemod <- run_model(
  active_treatment_strategy,
  placebo_strategy,
  cycles = 1, # Represents the "depth" of the decision tree
  cost = cost,
  effect = utility
)

# Print results
summary(res_heemod)
plot(res_heemod, type = "ce")

ERROR: Error in library(heemod): there is no package called ‘heemod’
